In [ ]:
!pip install transformers torch

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

In [ ]:
model_name = "microsoft/DialoGPT-medium"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

In [ ]:
chat_history_ids = None

print("Chatbot: Hello! I am your AI assistant. Type 'exit' to stop.")

while True:
    user_input = input("You: ")

    if user_input.lower() in ["exit", "quit"]:
        print("Chatbot: Goodbye! 👋")
        break

    # Encode input
    new_input_ids = tokenizer.encode(user_input + tokenizer.eos_token, return_tensors='pt')
    # Maintain conversation history
    attention_mask = torch.ones(new_input_ids.shape, dtype=torch.long)

    bot_input_ids = torch.cat([chat_history_ids, new_input_ids], dim=-1) if chat_history_ids is not None else new_input_ids

    # Generate response
    chat_history_ids = model.generate(
      bot_input_ids,
      max_length=1000,
      pad_token_id=tokenizer.eos_token_id,
      do_sample=True,
      temperature=0.7,   # control randomness
      top_k=40,
      top_p=0.9
    )

    # Decode response
    bot_response = tokenizer.decode(chat_history_ids[:, bot_input_ids.shape[-1]:][0], skip_special_tokens=True)

    print("Chatbot:", bot_response)